# Tools

## Define a tool with type hints and a docstring.

In [1]:
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for matching records.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

## Override the name/description, or define a rich input schema with Pydantic.

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

@tool("web_search", description="Search the web for information.")
def search(query: str) -> str:
    """Search the web."""
    return f"Results for: {query}"

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(default="celsius")


@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius") -> str:
    """Get current weather for a location."""
    temp = 22 if units == "celsius" else 72
    return f"{location}: {temp} degrees"

## Bind tools to a model, or hand them to an agent that runs the loop for you.

In [6]:
import os
from pydantic import SecretStr

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)
res = model.bind_tools([get_weather]).invoke("weather in Boston")
print(res.tool_calls)

agent = create_agent(
    model=model,
    tools=[search_database, get_weather],
    system_prompt="You are a helpful assistant."
)

result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Paris? Also get me first 10 weather details from the database"}]})
print(result["messages"][-1].content)

[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'chatcmpl-tool-bef06fdcd39d0138', 'type': 'tool_call'}]
The current temperature in Paris is **22 °C**.

I also queried the internal database for the first 10 records that contain “weather”. The query returned **10 results**. Those records include various weather‑related entries (e.g., past forecasts, temperature readings, humidity levels, etc.) from our system. If you’d like to see the specific content of any of those records, just let me know which ones you’re interested in!
